In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 3.0.6
NumPy: 2.5.3


In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = None

    for path in [CURRENT_DIR, *CURRENT_DIR.parents]:
        if (path / "data" / "processed").is_dir() and (path / "notebooks").is_dir():
            PROJECT_ROOT = path
            break

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            f"Could not locate H-IDS project root from: {CURRENT_DIR}"
        )

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"

SPLITS_DIR.mkdir(parents=True, exist_ok=True)

FINAL_DATA_PATH = (
    PROCESSED_DATA_DIR /
    "cic_ids2017_cleaned_deduplicated.parquet"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nFinal dataset:")
print(FINAL_DATA_PATH)

print("\nFinal dataset exists:", FINAL_DATA_PATH.exists())

Project root:
c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS

Final dataset:
c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS\data\processed\cic_ids2017_cleaned_deduplicated.parquet

Final dataset exists: True


In [3]:
labels_df = pd.read_parquet(
    FINAL_DATA_PATH,
    columns=["Label"]
)

print("Label data loaded successfully.")
print(f"Rows: {len(labels_df):,}")
print(f"Columns: {len(labels_df.columns)}")

Label data loaded successfully.
Rows: 2,520,798
Columns: 1


In [4]:
labels_df["row_id"] = np.arange(
    len(labels_df),
    dtype=np.int64
)

print(labels_df.head())
print(labels_df.tail())

    Label  row_id
0  BENIGN       0
1  BENIGN       1
2  BENIGN       2
3  BENIGN       3
4  BENIGN       4
          Label   row_id
2520793  BENIGN  2520793
2520794  BENIGN  2520794
2520795  BENIGN  2520795
2520796  BENIGN  2520796
2520797  BENIGN  2520797


In [5]:
label_counts = labels_df["Label"].value_counts()

print("FINAL DATASET CLASS DISTRIBUTION")
print("=" * 60)
print(label_counts)

FINAL DATASET CLASS DISTRIBUTION
Label
BENIGN                        2095057
DoS Hulk                       172846
DDoS                           128014
PortScan                        90694
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
Bot                              1948
Web Attack - Brute Force         1470
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [6]:
multiclass_labels = sorted(
    labels_df["Label"].unique()
)

multiclass_mapping = {
    label: class_id
    for class_id, label in enumerate(multiclass_labels)
}

print("MULTICLASS LABEL MAPPING")
print("=" * 60)

for label, class_id in multiclass_mapping.items():
    print(f"{class_id:2} -> {label}")

MULTICLASS LABEL MAPPING
 0 -> BENIGN
 1 -> Bot
 2 -> DDoS
 3 -> DoS GoldenEye
 4 -> DoS Hulk
 5 -> DoS Slowhttptest
 6 -> DoS slowloris
 7 -> FTP-Patator
 8 -> Heartbleed
 9 -> Infiltration
10 -> PortScan
11 -> SSH-Patator
12 -> Web Attack - Brute Force
13 -> Web Attack - Sql Injection
14 -> Web Attack - XSS


In [7]:
labels_df["binary_label"] = (
    labels_df["Label"] != "BENIGN"
).astype(np.int8)

print("BINARY LABEL DISTRIBUTION")
print("=" * 60)

print(
    labels_df["binary_label"].value_counts()
    .sort_index()
)

print("\nMapping:")
print("0 -> BENIGN")
print("1 -> ATTACK")

BINARY LABEL DISTRIBUTION
binary_label
0    2095057
1     425741
Name: count, dtype: int64

Mapping:
0 -> BENIGN
1 -> ATTACK


In [8]:
all_row_ids = labels_df["row_id"].to_numpy()
all_labels = labels_df["Label"].to_numpy()

train_val_ids, test_ids, train_val_labels, test_labels = train_test_split(
    all_row_ids,
    all_labels,
    test_size=0.10,
    random_state=42,
    stratify=all_labels
)

print("Train + Validation:", len(train_val_ids))
print("Test:", len(test_ids))

Train + Validation: 2268718
Test: 252080


In [9]:
train_ids, val_ids, train_labels, val_labels = train_test_split(
    train_val_ids,
    train_val_labels,
    test_size=1/9,
    random_state=42,
    stratify=train_val_labels
)

print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

Train: 2016638
Validation: 252080
Test: 252080


In [11]:
total_split_rows = (
    len(train_ids)
    + len(val_ids)
    + len(test_ids)
)

print("Total split rows:", total_split_rows)
print("Total dataset rows:", len(labels_df))

assert total_split_rows == len(labels_df)

Total split rows: 2520798
Total dataset rows: 2520798


In [12]:
train_set = set(train_ids)
val_set = set(val_ids)
test_set = set(test_ids)

print("Train ∩ Validation:", len(train_set & val_set))
print("Train ∩ Test:", len(train_set & test_set))
print("Validation ∩ Test:", len(val_set & test_set))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [13]:
# Save canonical split indices

train_split_df = pd.DataFrame({
    "row_id": train_ids
})

val_split_df = pd.DataFrame({
    "row_id": val_ids
})

test_split_df = pd.DataFrame({
    "row_id": test_ids
})

train_split_path = SPLITS_DIR / "train_indices.csv"
val_split_path = SPLITS_DIR / "validation_indices.csv"
test_split_path = SPLITS_DIR / "test_indices.csv"

train_split_df.to_csv(train_split_path, index=False)
val_split_df.to_csv(val_split_path, index=False)
test_split_df.to_csv(test_split_path, index=False)

print("Split indices saved successfully.")

print(f"Train:      {train_split_path}")
print(f"Validation: {val_split_path}")
print(f"Test:       {test_split_path}")

Split indices saved successfully.
Train:      c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS\data\splits\train_indices.csv
Validation: c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS\data\splits\validation_indices.csv
Test:       c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS\data\splits\test_indices.csv


In [14]:
import json
label_mapping_data = {
    "binary": {
        "0": "BENIGN",
        "1": "ATTACK"
    },
    "multiclass": {
        str(class_id): label
        for label, class_id in multiclass_mapping.items()
    }
}

LABEL_MAPPING_PATH = SPLITS_DIR / "label_mapping.json"

with open(LABEL_MAPPING_PATH, "w", encoding="utf-8") as f:
    json.dump(
        label_mapping_data,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Label mapping saved:")
print(LABEL_MAPPING_PATH)

Label mapping saved:
c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS\data\splits\label_mapping.json


In [15]:
split_metadata = {
    "dataset": "CIC-IDS2017",
    "dataset_file": "cic_ids2017_cleaned_deduplicated.parquet",
    "total_rows": int(len(labels_df)),
    "train_rows": int(len(train_ids)),
    "validation_rows": int(len(val_ids)),
    "test_rows": int(len(test_ids)),
    "train_ratio": 0.80,
    "validation_ratio": 0.10,
    "test_ratio": 0.10,
    "random_state": 42,
    "stratification": "15-class multiclass Label",
    "binary_mapping": {
        "0": "BENIGN",
        "1": "ATTACK"
    }
}

SPLIT_METADATA_PATH = SPLITS_DIR / "split_metadata.json"

with open(SPLIT_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(
        split_metadata,
        f,
        indent=4
    )

print("Split metadata saved:")
print(SPLIT_METADATA_PATH)

Split metadata saved:
c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS\data\splits\split_metadata.json


In [16]:
print("FILES IN data/splits/")
print("=" * 50)

for file in sorted(SPLITS_DIR.iterdir()):
    print(file.name)

FILES IN data/splits/
label_mapping.json
split_metadata.json
test_indices.csv
train_indices.csv
validation_indices.csv


In [17]:
# Class distribution across Train / Validation / Test

split_distribution = pd.DataFrame({
    "Train": pd.Series(train_labels).value_counts(),
    "Validation": pd.Series(val_labels).value_counts(),
    "Test": pd.Series(test_labels).value_counts()
}).fillna(0).astype(int)

split_distribution["Total"] = (
    split_distribution["Train"]
    + split_distribution["Validation"]
    + split_distribution["Test"]
)

split_distribution["Expected Total"] = (
    labels_df["Label"].value_counts()
)

split_distribution

,Train,Validation,Test,Total,Expected Total
BENIGN,1676045,209506,209506,2095057,2095057
DoS Hulk,138276,17285,17285,172846,172846
DDoS,102412,12801,12801,128014,128014
PortScan,72556,9069,9069,90694,90694
DoS GoldenEye,8228,1029,1029,10286,10286
FTP-Patator,4745,593,593,5931,5931
DoS slowloris,4308,539,538,5385,5385
DoS Slowhttptest,4182,523,523,5228,5228
SSH-Patator,2575,322,322,3219,3219
Bot,1558,195,195,1948,1948


In [18]:
# Percentage distribution within each split

split_percentages = split_distribution[
    ["Train", "Validation", "Test"]
].copy()

for column in split_percentages.columns:
    split_percentages[column] = (
        split_percentages[column]
        / split_percentages[column].sum()
        * 100
    )

split_percentages.round(4)

,Train,Validation,Test
BENIGN,83.1109,83.1109,83.1109
DoS Hulk,6.8568,6.8570,6.8570
DDoS,5.0784,5.0781,5.0781
PortScan,3.5979,3.5977,3.5977
DoS GoldenEye,0.4080,0.4082,0.4082
FTP-Patator,0.2353,0.2352,0.2352
DoS slowloris,0.2136,0.2138,0.2134
DoS Slowhttptest,0.2074,0.2075,0.2075
SSH-Patator,0.1277,0.1277,0.1277
Bot,0.0773,0.0774,0.0774


In [19]:
# Verify class totals are preserved

assert (
    split_distribution["Total"]
    == labels_df["Label"].value_counts()
).all()

print("Class totals preserved across all splits: PASS")

Class totals preserved across all splits: PASS
